# Práctica 1 – Ingeniería del Conocimiento · Fase 1: EDA
Réplica del código ejecutado en la Fase 1 (predicción de `price`). Ejecutar de arriba abajo en un Colab nuevo.

**Antes de empezar:** ten a mano `train-1.csv` y `test-1.csv` (la celda de carga te pedirá subirlos).

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

print(pd.__version__, np.__version__, sklearn.__version__, scipy.__version__, matplotlib.__version__)

## 0. Carga de datos (Colab)
Sube `train-1.csv` y `test-1.csv` con el cuadro de diálogo. Alternativa con Google Drive (descomenta y ajusta la ruta):

In [ ]:
from google.colab import files
uploaded = files.upload()   # selecciona train-1.csv y test-1.csv

# Alternativa con Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/RUTA/train-1.csv', 'train-1.csv')
# shutil.copy('/content/drive/MyDrive/RUTA/test-1.csv', 'test-1.csv')

In [ ]:
tr = pd.read_csv('train-1.csv'); te = pd.read_csv('test-1.csv')
train_orig, test_orig = tr.copy(), te.copy()   # copias intactas para la celda final

# Separación por tipo con select_dtypes (independiente de la versión de pandas)
feats = tr.drop(columns=['id', 'price'])
cat = feats.select_dtypes(exclude='number').columns.tolist()
num = feats.select_dtypes(include='number').columns.tolist()
print('cat:', cat)
print('num:', num)
# Esperado: cat = category, brand, brand_tier, material, season, channel, country
#           num = weight_kg, rating, reviews, shipping_cost, margin, discount, is_promo

## 1. Estructura, tipos y faltantes

In [ ]:
print(tr.shape, te.shape)
print(tr.dtypes)
print(tr.head())
print(tr.isna().sum()); print(te.isna().sum())

## 2. Precio, correlaciones y relación de las categóricas con el precio

In [ ]:
p=tr.price
print('--- price'); print(p.describe(percentiles=[.01,.05,.25,.5,.75,.9,.95,.99]).round(2))
print('skew',round(stats.skew(p),2),'kurt',round(stats.kurtosis(p),2),'log skew',round(stats.skew(np.log(p)),2))
for t in [300,400,500,700]: print('price>',t,(p>t).sum(), 'share of total SSE around mean', round((((p-p.mean())**2)[p>t]).sum()/((p-p.mean())**2).sum(),3))
print('--- duplicates', tr.drop(columns='id').duplicated().sum(), 'id unique', tr.id.is_unique, 'overlap', len(set(tr.id)&set(te.id)))
print(tr.id.min(),tr.id.max(),te.id.min(),te.id.max())
print('--- numeric summary'); print(tr[num+['price']].describe().T.round(3))
print('--- corr pearson/spearman')
for c in num: print(c, round(tr[c].corr(p),3), round(tr[c].corr(p,method='spearman'),3), 'log-price pearson', round(tr[c].corr(np.log(p)),3))
print('--- cat vs price')
for c in cat:
    g=tr.assign(**{c:tr[c].fillna('NA')}).groupby(c).price.agg(['count','mean','median','std']).round(1)
    print(g)
    groups=[x.price.values for _,x in tr.assign(**{c:tr[c].fillna('NA')}).groupby(c)]
    F,pv=stats.f_oneway(*groups); H,pk=stats.kruskal(*groups)
    print(c,'ANOVA F=%.1f p=%.2g | Kruskal p=%.2g'%(F,pv,pk))

## 3. Cola alta, peso, relaciones entre numéricas, faltantes de `material` e interacciones

In [ ]:
pd.set_option('display.width',200); pd.set_option('display.max_columns',30)
p=tr.price
print('--- cola alta: price>400 vs resto (medias)')
hi=tr[p>400]; print(hi[num].mean().round(3).to_frame('hi>400').join(tr[p<=400][num].mean().round(3).to_frame('resto')))
for c in ['category','brand','brand_tier']: print(hi[c].value_counts(normalize=True).round(2).to_dict())
print('weight_kg en cola alta: ', hi.weight_kg.describe().round(2).to_dict())
print('--- weight_kg por cuantiles vs price')
q=pd.qcut(tr.weight_kg,10); print(tr.groupby(q,observed=True).price.agg(['mean','median','std','max']).round(1))
print('weight>3: n',(tr.weight_kg>3).sum(),' price mean',tr[tr.weight_kg>3].price.mean().round(1))
print('--- relaciones entre numericas'); print(tr[num+['price']].corr().round(2))
print('--- price/weight y shipping/weight')
print(np.corrcoef(np.log(tr.weight_kg),np.log(p))[0,1], np.corrcoef(np.log(tr.weight_kg),tr.shipping_cost)[0,1])
print('--- rating, reviews distribucion'); print(tr[['rating','reviews']].describe().round(2)); print('rating==5',(tr.rating==5).sum(),'reviews>500',(tr.reviews>500).sum())
print('--- faltantes material vs resto'); m=tr.material.isna()
for c in ['category','brand_tier','channel','country','season']:
    print(c, (pd.crosstab(tr[c],m,normalize='index')[True]*100).round(1).to_dict())
print('t-test price NA vs no NA', stats.ttest_ind(p[m],p[~m],equal_var=False))
print('mean num NA vs no NA'); print(tr.groupby(m)[num].mean().round(3))
print('--- interaccion pendiente weight por categoria / tier (corr log-log)')
for c in ['category','brand_tier']:
    for k,g in tr.groupby(c): print(c,k, round(np.polyfit(np.log(g.weight_kg),np.log(g.price),1)[0],3))

## 4. Redundancias, heterocedasticidad y comparación train vs test (KS y χ²)

In [ ]:
pd.set_option('display.width',200); pd.set_option('display.max_columns',30)
print('--- discount vs is_promo'); print(tr.groupby('is_promo').discount.describe().round(3))
print('--- brand vs brand_tier'); print(pd.crosstab(tr.brand,tr.brand_tier))
print('--- heterocedasticidad: CV del precio por decil de weight (std/mean)')
q=pd.qcut(tr.weight_kg,10); g=tr.groupby(q,observed=True).price; print((g.std()/g.mean()).round(2).values)
print('--- test vs train: numericas (KS) y medias')
rows=[]
for c in num:
    ks=stats.ks_2samp(tr[c],te[c]); rows.append((c,round(tr[c].mean(),3),round(te[c].mean(),3),round(tr[c].median(),3),round(te[c].median(),3),round(tr[c].max(),2),round(te[c].max(),2),round(ks.statistic,4),round(ks.pvalue,3)))
print(pd.DataFrame(rows,columns=['var','mean_tr','mean_te','med_tr','med_te','max_tr','max_te','KS','p']))
print('--- test vs train: categoricas (chi2)')
for c in cat:
    a=tr[c].fillna('NA').value_counts(normalize=True); b=te[c].fillna('NA').value_counts(normalize=True)
    ct=pd.concat([tr[c].fillna('NA').value_counts(),te[c].fillna('NA').value_counts()],axis=1).fillna(0)
    chi,pv,_,_=stats.chi2_contingency(ct.values)
    print(c,'chi2 p=%.3f'%pv,'| cats solo en test:',set(te[c].dropna())-set(tr[c].dropna()),'| max dif prop %.3f'%(a.sub(b,fill_value=0).abs().max()))
print('--- rango test dentro de train?')
for c in ['weight_kg','shipping_cost','reviews']: print(c,'test>max_train',(te[c]>tr[c].max()).sum(),'| test>p99 train',(te[c]>tr[c].quantile(.99)).mean().round(4),'| train>p99 train',(tr[c]>tr[c].quantile(.99)).mean().round(4))
print('weight_kg>3: train %.3f test %.3f'%((tr.weight_kg>3).mean(),(te.weight_kg>3).mean()))

## 5. Proporciones train/test, deriva por `id` y validación adversaria (semillas fijas)
Nota: esta celda añade la columna `id_bloque` al DataFrame `tr` (igual que en la ejecución original).

In [ ]:
warnings.filterwarnings('ignore')
pd.set_option('display.width',200)
for c in ['category','channel','material']:
    print(pd.concat([tr[c].fillna('NA').value_counts(normalize=True).round(3).rename('train'),te[c].fillna('NA').value_counts(normalize=True).round(3).rename('test')],axis=1))
# deriva con id (train ordenado)
tr['id_bloque']=pd.cut(tr.id,8,labels=False)
print('--- precio medio por bloque de id (train)'); print(tr.groupby('id_bloque').price.agg(['mean','median','std']).round(1).T)
print('corr id-price',round(tr.id.corr(tr.price),3))
print('channel online por bloque de id',tr.groupby('id_bloque').channel.apply(lambda s:(s=='Online').mean()).round(3).values)
print('category Electronica por bloque',tr.groupby('id_bloque').category.apply(lambda s:(s=='Electrónica').mean()).round(3).values)
# adversarial (sin id)
X=pd.concat([tr[cat+num],te[cat+num]],ignore_index=True)
for c in cat: X[c]=X[c].fillna('NA').astype('category')
y=np.r_[np.zeros(len(tr)),np.ones(len(te))]
aucs=[]
for seed in [0,1,2]:
    clf=HistGradientBoostingClassifier(max_iter=150,learning_rate=0.05,max_depth=4,categorical_features='from_dtype',random_state=seed)
    pr=cross_val_predict(clf,X,y,cv=StratifiedKFold(5,shuffle=True,random_state=seed),method='predict_proba')[:,1]
    aucs.append(roc_auc_score(y,pr))
print('--- AUC adversarial (sin id), 3 semillas:',np.round(aucs,3),'media',round(np.mean(aucs),3))
# permutation: AUC con permutacion de etiquetas (referencia nula)
rng=np.random.RandomState(0); nulls=[]
for s in range(3):
    yp=rng.permutation(y); clf=HistGradientBoostingClassifier(max_iter=150,learning_rate=0.05,max_depth=4,categorical_features='from_dtype',random_state=0)
    pr=cross_val_predict(clf,X,yp,cv=StratifiedKFold(5,shuffle=True,random_state=s),method='predict_proba')[:,1]; nulls.append(roc_auc_score(yp,pr))
print('AUC con etiquetas permutadas (nulo):',np.round(nulls,3))
clf=HistGradientBoostingClassifier(max_iter=150,learning_rate=0.05,max_depth=4,categorical_features='from_dtype',random_state=0).fit(X,y)
imp=permutation_importance(clf,X,y,scoring='roc_auc',n_repeats=5,random_state=0)
print(pd.Series(imp.importances_mean,index=X.columns).sort_values(ascending=False).round(4).head(8))
# chi2 conjunto category x channel
print('--- 8 contrastes train/test: p<0.05/ (15 contrastes) bonferroni =',round(0.05/14,4))

## 6. Gráficos
Se guardan en la carpeta `fig_eda/` y se muestran en el notebook. En estas celdas `num6` son las 6 numéricas continuas (sin `is_promo`), como en la ejecución original.

In [ ]:
os.makedirs('fig_eda', exist_ok=True)
o='fig_eda/'
p=tr.price
num6=[c for c in num if c!='is_promo']
# 1 distribución precio
f,ax=plt.subplots(1,3,figsize=(16,4))
ax[0].hist(p,bins=80,color='#2b6cb0'); ax[0].set_title('price (escala original)'); ax[0].axvline(p.mean(),c='r',ls='--',label='media'); ax[0].axvline(p.median(),c='k',ls=':',label='mediana'); ax[0].legend()
ax[1].hist(np.log(p),bins=80,color='#2f855a'); ax[1].set_title('log(price)')
ax[2].boxplot(p,vert=False,showfliers=True); ax[2].set_title('boxplot price')
plt.tight_layout(); plt.savefig(o+'01_precio.png',dpi=110); plt.show(); plt.close()
# 2 cola: contribución al error
s=np.sort(p.values)[::-1]; sse=(s-p.mean())**2; cum=np.cumsum(sse)/sse.sum()
f,ax=plt.subplots(figsize=(6,4)); ax.plot(np.arange(1,len(s)+1)/len(s)*100,cum); ax.set_xscale('log'); ax.set_xlabel('% de productos más caros (log)'); ax.set_ylabel('fracción de la varianza total'); ax.set_title('Concentración de la varianza en la cola'); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(o+'02_cola_varianza.png',dpi=110); plt.show(); plt.close()
# 3 numéricas vs precio
f,ax=plt.subplots(2,3,figsize=(16,8))
for a,c in zip(ax.ravel(),num6):
    a.scatter(tr[c],p,s=3,alpha=.25); a.set_xlabel(c); a.set_ylabel('price')
    if c in('weight_kg','reviews'): a.set_xscale('log')
    a.set_title(f'{c} (r={tr[c].corr(p):.2f})')
plt.tight_layout(); plt.savefig(o+'03_numericas_vs_precio.png',dpi=110); plt.show(); plt.close()

In [ ]:
warnings.filterwarnings('ignore')
o='fig_eda/'
p=tr.price
num6=[c for c in num if c!='is_promo']
# 4 categóricas vs precio
f,ax=plt.subplots(2,4,figsize=(20,8)); t=tr.assign(material=tr.material.fillna('NA'))
for a,c in zip(ax.ravel(),cat):
    gs=list(t.groupby(c)); a.boxplot([x.price.values for _,x in gs],tick_labels=[k for k,_ in gs],showfliers=False); a.set_title(c); a.tick_params(axis='x',rotation=30)
ax.ravel()[-1].axis('off'); plt.tight_layout(); plt.savefig(o+'04_categoricas_vs_precio.png',dpi=110); plt.show(); plt.close()
# 5 correlación
f,ax=plt.subplots(figsize=(7,6)); cm=tr[num6+['is_promo','price']].corr(); im=ax.imshow(cm,cmap='coolwarm',vmin=-1,vmax=1)
ax.set_xticks(range(len(cm))); ax.set_xticklabels(cm.columns,rotation=45,ha='right'); ax.set_yticks(range(len(cm))); ax.set_yticklabels(cm.columns)
for i in range(len(cm)):
    for j in range(len(cm)): ax.text(j,i,f'{cm.iloc[i,j]:.2f}',ha='center',va='center',fontsize=7)
plt.colorbar(im); plt.tight_layout(); plt.savefig(o+'05_correlacion.png',dpi=110); plt.show(); plt.close()
# 6 train vs test numéricas
f,ax=plt.subplots(2,3,figsize=(16,7))
for a,c in zip(ax.ravel(),num6):
    lo,hi=tr[c].quantile(.001),tr[c].quantile(.999); b=np.linspace(lo,hi,40)
    a.hist(tr[c],bins=b,alpha=.5,density=True,label='train'); a.hist(te[c],bins=b,alpha=.5,density=True,label='test'); a.set_title(c); a.legend()
plt.tight_layout(); plt.savefig(o+'06_train_vs_test_num.png',dpi=110); plt.show(); plt.close()
# 7 train vs test categóricas
f,ax=plt.subplots(2,4,figsize=(20,8))
for a,c in zip(ax.ravel(),cat):
    d=pd.concat([tr[c].fillna('NA').value_counts(normalize=True).rename('train'),te[c].fillna('NA').value_counts(normalize=True).rename('test')],axis=1).sort_index()
    d.plot.bar(ax=a); a.set_title(c); a.set_xlabel('')
ax.ravel()[-1].axis('off'); plt.tight_layout(); plt.savefig(o+'07_train_vs_test_cat.png',dpi=110); plt.show(); plt.close()
print('ok')

## 7. Valores truncados (suelos/techos)

In [ ]:
for n,d in [('train',tr),('test',te)]:
    print(n,'rating==5:',(d.rating==5).sum(),'| margin==0.05:',(d.margin==0.05).sum(),'| discount==0:',(d.discount==0).sum(),'| shipping==0:',(d.shipping_cost==0).sum(),'| reviews==3:',(d.reviews==3).sum())
print('is_promo=0 & discount>0.18:',((tr.is_promo==0)&(tr.discount>0.18)).sum(),'| is_promo=1 & discount<0.18:',((tr.is_promo==1)&(tr.discount<0.18)).sum())

## 8. Números clave para comparar con `decisiones.md`
(Celda de resumen nueva, añadida para la comparación; usa copias intactas de los datos cargados.)

In [ ]:
a, b = train_orig, test_orig
p = a.price
print('=== Dimensiones ===')
print('train:', a.shape[0], 'filas x', a.shape[1], 'columnas | test:', b.shape[0], 'filas x', b.shape[1], 'columnas')
print('=== Faltantes ===')
print('train:', a.isna().sum()[a.isna().sum() > 0].to_dict(), '| test:', b.isna().sum()[b.isna().sum() > 0].to_dict())
print('material faltante: train %.1f %% | test %.1f %%' % (a.material.isna().mean()*100, b.material.isna().mean()*100))
print('=== price ===')
print('media %.1f | mediana %.1f | std %.1f | min %.2f | max %.2f' % (p.mean(), p.median(), p.std(), p.min(), p.max()))
print('asimetria %.2f | curtosis %.2f | asimetria de log(price) %.2f' % (stats.skew(p), stats.kurtosis(p), stats.skew(np.log(p))))
print('price>400:', int((p > 400).sum()), '| price>500:', int((p > 500).sum()))
print('=== Correlacion Pearson con price ===')
print(a[num].corrwith(p).round(2).sort_values(ascending=False).to_dict())
print('=== Otros ===')
print('id unico:', a.id.is_unique, '| solapamiento train-test:', len(set(a.id) & set(b.id)), '| duplicados (sin id):', int(a.drop(columns='id').duplicated().sum()))